<hr style="border: 6px solid#003262;" />

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/03_cover.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

<br>

# BAYESIAN HYPERPARAMETER OPTIMIZATION

<br>

**About:** A conceptual and practical guide to Bayesian optimization as a smarter alternative to grid and random search for tuning model hyperparameters - explaining the surrogate model and acquisition function mechanism, then implementing it with Optuna on a real classification task.

**Learning Goals:**
* Understand the hyperparameter optimization problem as black-box function optimization
* Compare Grid Search, Random Search, and Bayesian optimization by efficiency and coverage
* Explain how a surrogate model (Gaussian Process) approximates the objective function
* Understand acquisition functions and why they balance exploration vs. exploitation
* Implement Bayesian optimization with Optuna on a sklearn classifier
* Visualize the search history and interpret which hyperparameters matter most
* Know when Bayesian optimization is worth the added complexity over random search

**Keywords:** Bayesian optimization, Gaussian process, acquisition function, hyperparameter tuning, Optuna, random search, grid search, exploration vs. exploitation

**Prerequisite Knowledge:** (1) Python and scikit-learn basics, (2) Binary classification and AUC metric, (3) Familiarity with RandomizedSearchCV from Notebook 01 is helpful but not required

**Target User:** ML practitioners who have used random search before and want to understand both why it works and when Bayesian optimization delivers a meaningful improvement.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SETUP AND PROBLEM FRAMING](#Part_0)
> #### [PART 1: GRID SEARCH AND RANDOM SEARCH BASELINES](#Part_1)
> #### [PART 2: BAYESIAN OPTIMIZATION - THEORY](#Part_2)
> #### [PART 3: BAYESIAN OPTIMIZATION - IMPLEMENTATION WITH OPTUNA](#Part_3)
> #### [PART 4: COMPARING ALL THREE STRATEGIES](#Part_4)

<br>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import randint, loguniform, norm

# Optuna for Bayesian optimization
# Install with: pip install optuna
# TODO: verify optuna API against current docs at https://optuna.readthedocs.io (verified 2026-08)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** and Problem Framing

<a id='Part_0_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 0.1: THE HYPERPARAMETER OPTIMIZATION PROBLEM

<br>

Training a machine learning model involves two distinct kinds of optimization:

1. **Inner optimization (training):** Minimize the training loss by adjusting model weights (gradient descent, L-BFGS, etc.). This is done by the training algorithm automatically.

2. **Outer optimization (hyperparameter search):** Find the settings - learning rate, regularization strength, number of trees, depth, dropout - that make the trained model perform best on held-out data. This is **not** done by the training algorithm; it requires a separate search strategy.

The outer optimization problem has three properties that make it hard:

- **Black-box:** We cannot compute the gradient of validation AUC with respect to hyperparameters. Each evaluation requires training a model from scratch.
- **Expensive:** Each evaluation may take minutes or hours.
- **Noisy:** Validation AUC has variance across random seeds, data splits, and initialization.

These properties explain why exhaustive grid search is impractical: we cannot afford to evaluate every combination, and the function landscape is not smooth enough for gradient-based optimization.

___

**Sources Consulted:**
- Shahriari et al., "Taking the Human Out of the Loop: A Review of Bayesian Optimization," Proceedings of the IEEE (2016) - primary survey; stable reference for the problem framing
- Bergstra and Bengio, "Random Search for Hyper-Parameter Optimization," JMLR (2012) - stable reference for random search theory

___

In [ ]:
# Load data and set up a consistent benchmark task
bc = load_breast_cancer()
X, y = bc.data, bc.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Dataset: Breast Cancer Wisconsin')
print(f'Features: {X.shape[1]}, Samples: {X.shape[0]}')
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')
print(f'Positive rate (train): {y_train.mean():.3f}')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **GRID SEARCH** and Random Search Baselines

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: GRID SEARCH

<br>

Grid Search evaluates every combination of hyperparameter values from a pre-specified grid. For a 5-parameter grid with 3 values each, that is $3^5 = 243$ evaluations. With 3-fold cross-validation, the estimator is trained $243 \times 3 = 729$ times.

**Key limitation:** Grid Search wastes evaluations on unimportant hyperparameters. If the learning rate matters but the subsample fraction does not, Grid Search still evaluates every combination of subsample values for each learning rate value. As Bergstra and Bengio showed, most of the performance variance in typical ML models comes from 1-2 hyperparameters - Grid Search spreads its budget uniformly regardless of importance.

The grid below uses a conservative range and coarse resolution to keep runtime manageable for this demonstration.

In [ ]:
# Grid search on a Gradient Boosting classifier
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 4],
    'learning_rate': [0.05, 0.1, 0.2],
}

gb = GradientBoostingClassifier(random_state=42)

t0 = time.time()
grid_search = GridSearchCV(
    gb,
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

n_evals_grid = len(grid_search.cv_results_['mean_test_score'])
print(f'Grid search: {n_evals_grid} parameter combinations, {n_evals_grid * 3} total fits')
print(f'Time: {grid_time:.1f}s')
print(f'Best params: {grid_search.best_params_}')
print(f'Best CV AUC: {grid_search.best_score_:.4f}')
test_auc_grid = roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1])
print(f'Test AUC: {test_auc_grid:.4f}')

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: RANDOM SEARCH

<br>

Random Search samples each hyperparameter independently from its distribution. The key insight: if you have 10 hyperparameters but only 2 drive most of the variance, any single row or column of a random sample is essentially a unique setting of those 2 important parameters. Grid Search evaluates all combinations, but each is a duplicate on the unimportant dimensions.

With the same evaluation budget as Grid Search, Random Search tends to cover the important hyperparameter dimensions more broadly - especially when they are continuous. We use log-uniform distributions for scale hyperparameters (learning rate, regularization strength) because the effect of doubling from 0.001 to 0.002 is similar in magnitude to doubling from 0.1 to 0.2.

In [ ]:
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 8),
    'learning_rate': loguniform(0.01, 0.5),
    'subsample': loguniform(0.5, 1.0),        # added 4th dim, impossible to grid-search cheaply
    'min_samples_leaf': randint(1, 20),        # added 5th dim
}

# Match the number of evaluations to grid search for a fair budget comparison
n_iter = n_evals_grid

t0 = time.time()
random_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=n_iter,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)
random_time = time.time() - t0

print(f'Random search: {n_iter} evaluations, 5 hyperparameters')
print(f'Time: {random_time:.1f}s')
print(f'Best params: {random_search.best_params_}')
print(f'Best CV AUC: {random_search.best_score_:.4f}')
test_auc_random = roc_auc_score(y_test, random_search.predict_proba(X_test)[:, 1])
print(f'Test AUC: {test_auc_random:.4f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Grid Search over the 3-hyperparameter grid above produces 18 combinations. Random Search with `n_iter=18` covers 5 hyperparameters including two Grid Search did not explore (subsample and min_samples_leaf). Extend the Grid Search grid to include the 5th hyperparameter `min_samples_leaf: [1, 5, 10]`. How many total combinations does the expanded grid have? At 3-fold CV, how many model fits does that require? How does this scale if you add a 6th hyperparameter with 3 values?**

<br>

```python
# Extended grid
extended_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 4],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'min_samples_leaf': [1, 5, 10],
}
# Count combinations
n_combinations = ...
print(f'Combinations: {n_combinations}')
print(f'Total fits with 3-fold CV: {n_combinations * 3}')
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **BAYESIAN OPTIMIZATION** - Theory

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: THE SURROGATE MODEL

<br>

Bayesian optimization uses the history of past evaluations to build a **surrogate model** - a cheap approximation of the expensive objective function. The surrogate predicts not just the expected performance at an untried hyperparameter setting, but also the **uncertainty** in that prediction.

**Gaussian Process (GP)** is the classical surrogate for Bayesian optimization. A GP defines a distribution over functions, parameterized by a mean function and a covariance kernel. After observing $n$ evaluations, the GP posterior gives:
- A predicted mean performance $\mu(\lambda)$ at any candidate $\lambda$
- A predicted standard deviation $\sigma(\lambda)$ representing uncertainty

Near observed points, $\sigma(\lambda) \approx 0$ (we know what happens there). In unexplored regions, $\sigma(\lambda)$ is large, signaling high uncertainty.

**Tree-structured Parzen Estimator (TPE)**, used by Optuna, is a different surrogate that models $p(\lambda | y > y^*)$ and $p(\lambda | y \leq y^*)$ separately (where $y^*$ is a threshold) and selects new candidates where the ratio of the two densities is highest. TPE scales better to high-dimensional categorical spaces than GP does.

___

**Sources Consulted:**
- Bergstra et al., "Algorithms for Hyper-Parameter Optimization," NeurIPS (2011) - original TPE paper; stable
- Shahriari et al., "Taking the Human Out of the Loop," IEEE (2016) - stable survey for GP-based Bayesian optimization

___

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: ACQUISITION FUNCTIONS

<br>

The surrogate alone does not tell us where to sample next. An **acquisition function** uses the surrogate's posterior to score candidate points, balancing two competing goals:

- **Exploitation:** Sample near the current best observation, where the mean $\mu(\lambda)$ is high.
- **Exploration:** Sample in high-uncertainty regions, where $\sigma(\lambda)$ is large and we might find something much better.

**Expected Improvement (EI)** is the most common acquisition function:

$$EI(\lambda) = \mathbb{E}\left[\max(f(\lambda) - f^+, 0)\right]$$

where $f^+ = \max_{i \leq t} f(\lambda_i)$ is the best observed value so far. EI integrates over the GP posterior to compute the expected amount by which a new evaluation at $\lambda$ exceeds $f^+$. It naturally balances exploration and exploitation: uncertain regions with high $\sigma(\lambda)$ have higher EI even if their mean $\mu(\lambda)$ is modest.

**Upper Confidence Bound (UCB)** is simpler to interpret:

$$UCB(\lambda) = \mu(\lambda) + \kappa \cdot \sigma(\lambda)$$

A higher $\kappa$ emphasizes exploration; lower $\kappa$ emphasizes exploitation. Unlike EI, UCB has an explicit trade-off knob.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The plot above shows the GP posterior and Expected Improvement. After observing the four points, the acquisition function suggests a new evaluation at approximately x=0.45. Add a fifth observation at that point with value y=0.85 (discovered it is very good) and re-plot. How does the surrogate change? Where does EI suggest sampling next?**

<br>

```python
x_obs_new = np.append(x_obs, 0.45)
y_obs_new = np.append(y_obs, 0.85)
# Recompute GP posterior with the new observation and re-plot
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **BAYESIAN OPTIMIZATION** - Implementation with Optuna

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: SETTING UP AN OPTUNA STUDY

<br>

Optuna implements Bayesian optimization using the Tree-structured Parzen Estimator (TPE) by default, which works well for mixed (continuous + discrete + categorical) search spaces. The user defines an **objective function** that:
1. Receives a `trial` object
2. Samples hyperparameters from the trial using `trial.suggest_*` calls
3. Trains and evaluates a model
4. Returns a scalar score

Optuna then manages the search: after each evaluation it updates its internal model and suggests the next candidate. The `direction='maximize'` argument tells Optuna to maximize the returned score (AUC in our case).

The key difference from `RandomizedSearchCV`: instead of sampling all `n_iter` configurations upfront and evaluating them, Optuna evaluates one configuration, updates its surrogate, then decides where to sample next.

In [ ]:
def objective(trial):
    # Optuna objective: sample hyperparameters, train, return validation AUC
    # Suggest hyperparameters from the trial
    n_estimators = trial.suggest_int('n_estimators', 50, 400)
    max_depth = trial.suggest_int('max_depth', 2, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.5, log=True)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 30)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    gb = GradientBoostingClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Cross-validate on training data (3-fold for speed)
    scores = cross_val_score(gb, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

# Create and run the study
n_trials = n_evals_grid  # match budget to grid/random search for fair comparison

t0 = time.time()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
bayes_time = time.time() - t0

print(f'Bayesian optimization: {n_trials} trials')
print(f'Time: {bayes_time:.1f}s')
print(f'Best params: {study.best_params}')
print(f'Best CV AUC: {study.best_value:.4f}')

# Evaluate best params on test set
best_gb = GradientBoostingClassifier(**study.best_params, random_state=42)
best_gb.fit(X_train, y_train)
test_auc_bayes = roc_auc_score(y_test, best_gb.predict_proba(X_test)[:, 1])
print(f'Test AUC: {test_auc_bayes:.4f}')

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: VISUALIZING THE SEARCH HISTORY

<br>

In [ ]:
# Plot the optimization history (AUC vs. trial number)
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)

plt.figure(figsize=(9, 4))
plt.scatter(range(len(trial_values)), trial_values, alpha=0.5, s=20, color='#3B7EA1', label='Trial AUC')
plt.plot(range(len(best_so_far)), best_so_far, color='#003262', linewidth=2, label='Best so far')
plt.xlabel('Trial number')
plt.ylabel('Cross-validated AUC')
plt.title('Bayesian Optimization: Search History')
plt.legend()
plt.tight_layout()
plt.show()

# Hyperparameter importance (using Optuna's built-in importance analysis)
# TODO: verify optuna.importance API at https://optuna.readthedocs.io/en/stable/reference/importance.html
try:
    importance = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame(list(importance.items()), columns=['hyperparameter', 'importance'])
    imp_df = imp_df.sort_values('importance', ascending=True)

    plt.figure(figsize=(7, 4))
    plt.barh(imp_df['hyperparameter'], imp_df['importance'], color='#003262')
    plt.xlabel('Importance score (fraction of variance explained)')
    plt.title('Hyperparameter Importance (Optuna FAnova)')
    plt.tight_layout()
    plt.show()
    print('Most important hyperparameter:', imp_df.iloc[-1]['hyperparameter'])
except Exception as e:
    print(f'Importance analysis unavailable: {e}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The optimization history plot shows individual trial AUC (scattered) and the best-so-far curve. In a well-functioning Bayesian search, the best-so-far curve should improve faster in the first half (warmup phase) than the second half (refinement phase). Compute the improvement in best-so-far AUC between trials 0-5 and trials 5-n_trials. Does the improvement slow down? What does this tell you about diminishing returns with more evaluations?**

<br>

```python
# trial_values and best_so_far are available from the cell above
improvement_first_half = best_so_far[len(best_so_far)//2] - best_so_far[0]
improvement_second_half = best_so_far[-1] - best_so_far[len(best_so_far)//2]
print(f'Improvement first half: {improvement_first_half:.4f}')
print(f'Improvement second half: {improvement_second_half:.4f}')
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **COMPARING** All Three Strategies

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: FINAL COMPARISON

<br>

In [ ]:
# Summary comparison: same evaluation budget, three strategies
results = pd.DataFrame([
    {'Strategy': 'Grid Search',   'n_evals': n_evals_grid, 'n_hyperparams': 3, 'CV AUC': grid_search.best_score_,   'Test AUC': test_auc_grid,   'Time (s)': grid_time},
    {'Strategy': 'Random Search', 'n_evals': n_iter,       'n_hyperparams': 5, 'CV AUC': random_search.best_score_, 'Test AUC': test_auc_random, 'Time (s)': random_time},
    {'Strategy': 'Bayesian (TPE)','n_evals': n_trials,     'n_hyperparams': 6, 'CV AUC': study.best_value,          'Test AUC': test_auc_bayes,  'Time (s)': bayes_time},
])
print(results.to_string(index=False))

# Plot test AUC comparison
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(results['Strategy'], results['Test AUC'], color=['#C4820E', '#3B7EA1', '#003262'])
ax.set_ylim(results['Test AUC'].min() - 0.02, 1.01)
ax.set_ylabel('Test AUC')
ax.set_title(f'Hyperparameter Search Strategy Comparison ({n_evals_grid} evaluations each)')
for bar, val in zip(bars, results['Test AUC']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

<a id='Part_4_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.2: WHEN TO USE EACH STRATEGY

<br>

**Grid Search** - use when:
- You have at most 2-3 hyperparameters with small, discrete candidate sets
- You need reproducible, fully enumerated results
- The evaluation budget is large enough to cover all combinations

**Random Search** - use when:
- You have 4+ hyperparameters with continuous or large ranges
- You want a strong baseline with minimal implementation overhead
- Evaluations are expensive and you want to cover the space quickly

**Bayesian optimization** - use when:
- You have 5-20 hyperparameters
- Each evaluation is expensive (minutes to hours per trial)
- You are doing many searches and the upfront cost of implementing Bayesian optimization amortizes across projects
- You want principled uncertainty estimates about which regions of the search space are worth exploring

<strong style="color:red">KEY CONSIDERATION:</strong> Bayesian optimization is not always better than random search with the same budget. For cheap-to-evaluate objectives and small budgets (under 20 trials), the overhead of fitting the surrogate model can mean random search reaches a better solution first. The advantage of Bayesian optimization is asymptotic - with enough trials, it reliably outperforms random search on most objective surfaces.

___

**Sources Consulted:**
- Li et al., "Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization," JMLR (2018) - stable reference for adaptive budget allocation
- [Optuna documentation](https://optuna.readthedocs.io/) - TPE implementation details, verified 2026-08

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Run Bayesian optimization with `n_trials=5` and again with `n_trials=50`. Compare the best CV AUC found in both runs. At what budget does Bayesian optimization start to clearly outperform Random Search with the same budget? Write a loop that runs both strategies at budgets [5, 10, 20, 50, 100] and plots the best AUC found vs. budget for each strategy.**

<br>

```python
budgets = [5, 10, 20, 50]
bayes_aucs = []
random_aucs = []
for n in budgets:
    # Run Bayesian search with n trials
    study_n = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study_n.optimize(objective, n_trials=n)
    bayes_aucs.append(study_n.best_value)
    # Run random search with n iterations
    rs_n = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_dist, n_iter=n, cv=3, scoring='roc_auc', random_state=42)
    rs_n.fit(X_train, y_train)
    random_aucs.append(rs_n.best_score_)
# Plot
...
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<hr style="border: 6px solid#003262;" />